In [42]:
import sys
import json
import pandas as pd
import hashlib
import glob
import os
import shutil
import importlib.util
import filecmp

# https://pyadomd.readthedocs.io/
# pip install pyadomd
# https://learn.microsoft.com/en-us/analysis-services/client-libraries


# Ensure the ADOMD.NET path is in the system path prior to importing Pyadomd
from sys import path
adomd_path = r'C:\Program Files\Microsoft.NET\ADOMD.NET\160'

if not os.path.isdir(adomd_path):
    print("Folder does not exist.")
    sys.exit(1)

# Check if the ADOMD.NET path is already in the system path
if adomd_path not in path:
    path.append(adomd_path)

working_directory = os.getcwd()

# Check if the pyadomd module is installed and get its folder
# If it is installed, copy the pyadomd.txt file to the module's folder
# A common folder that it is installed in is C:\Users\username\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyadomd
spec = importlib.util.find_spec("pyadomd")
if spec and spec.origin and len(spec.submodule_search_locations) > 0:
    pyadomd_folder = spec.submodule_search_locations[0]
    src_file = "pyadomd.txt"  # Path to your source file (adjust if needed)
    # src_file = os.path.join(working_directory, "pyadomd.txt")
    dst_file = os.path.join(pyadomd_folder, "pyadomd.py")
    if not filecmp.cmp(src_file, dst_file, shallow=False):
        shutil.copy(src_file, dst_file)
else:
    print("pyadomd not found")
    sys.exit(1)

# Import Pyadomd after ensuring the path is set
from pyadomd import Pyadomd

def find_visual_id(id_map, start_id):
    current_id = start_id
    while current_id:
        node = id_map.get(current_id)
        if not node:
            break
        metrics = node.get('metrics', {})
        visual_id = metrics.get('visualId')
        if visual_id:
            return visual_id
        current_id = node.get('parentId')
    return None

# Flatten metrics into top-level columns
def flatten_event(event):
    flat = event.copy()
    metrics = flat.pop("metrics", {})
    flat.update(metrics)
    return flat

def row_hash(row):
    # Concatenate all values as string and hash
    row_str = '|'.join(str(val) for val in row.values)
    return hashlib.sha256(row_str.encode('utf-8')).hexdigest()

def find_visual_json_by_id(report_folder, visual_id):
    # Recursively search for all visual.json files
    for root, dirs, files in os.walk(report_folder):
        for file in files:
            if file == "visual.json":
                file_path = os.path.join(root, file)
                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        data = json.load(f)
                        if data.get("name") == visual_id:
                            return file_path
                except Exception as e:
                    print(f"Error reading {file_path}: {e}")
    return None

def get_page_display_name(visual_json_path):
    # Go up three directories to get to the page.json location
    page_dir = os.path.dirname(os.path.dirname(os.path.dirname(visual_json_path)))
    page_json_path = os.path.join(page_dir, "page.json")
    if os.path.isfile(page_json_path):
        with open(page_json_path, "r", encoding="utf-8") as f:
            page_data = json.load(f)
            return page_data.get("displayName")
    return None

# Assuming you have a DataFrame `filtered_df` with a 'visualId' column
def add_page_names_to_df(df, report_folder):
    page_names = []
    for visual_id in df['visualId']:
        visual_json_path = find_visual_json_by_id(report_folder, visual_id)
        if visual_json_path:
            page_name = get_page_display_name(visual_json_path)
        else:
            page_name = None
        page_names.append(page_name)
    df['pageName'] = page_names
    return df

project_name = "Sales & Returns Sample"
pbi_pa_folder_name = "PBI PA Files"
baseline_folder_name = "baseline"
baseline_parquet_file_name = "baseline.parquet"
baseline_csv_file_name = "baseline.csv"
output_csv_file = ""
output_parquet_file = ""
instance_folder = ""
instance_csv_file = ""
instance_parquet_file = ""
is_baseline = False  # Set to True if this run is a baseline, False otherwise
project_folder = os.path.join(working_directory, project_name)

# Connection string to your Power BI semantic model (update as needed)
connection_string = "Provider=MSOLAP;Data Source=localhost:56398;Initial Catalog=09a6b72e-64db-45a4-a6cc-264e4c88893a"

pbi_report_folder = os.path.join(r"C:\Users\Cory.Cundy\OneDrive - Core BTS\Desktop\Sales & Returns Sample v201912\Sales & Returns Sample v201912.Report")
# Create the project folder if it doesn't exist

# Specify the folder containing your JSON files
pbi_pa_folder = os.path.join(project_folder, pbi_pa_folder_name)
baseline_folder = os.path.join(project_folder, baseline_folder_name)
baseline_csv_file = os.path.join(baseline_folder, baseline_csv_file_name)
baseline_parquet_file = os.path.join(baseline_folder, baseline_parquet_file_name)

baseline_exists = os.path.isfile(baseline_parquet_file)
user_value = input("Press enter to create a baseline, otherwise enter a instance name: ")

# If the user presses enter, create a baseline folder if necessary
# If the user enters a value, create an instance folder if necessary
if user_value.strip() == "":
    instance_name = baseline_folder_name
    is_baseline = True

    if baseline_exists:
        overwrite = input("Do you want to overwrite the baseline (Y/N)? ").strip().lower() 
        if overwrite != 'y':
            print("Exiting without creating a new baseline.")
            sys.exit()
    else:
        print("Creating a new baseline folder.")
        os.makedirs(baseline_folder, exist_ok=True)
        print("Baseline folder created.")
else:
    is_baseline = False
    if not baseline_exists:
        print("Baseline folder does not exist. Please create a baseline first.")
        sys.exit()
    else:
        instance_name = user_value.strip()
        instance_folder = os.path.join(project_folder, instance_name)
        instance_csv_file = os.path.join(instance_folder, f"{instance_name}.csv")
        instance_parquet_file = os.path.join(instance_folder, f"{instance_name}.parquet")
        instance_exists = os.path.isfile(instance_parquet_file)

        if instance_exists:
            overwrite = input("Do you want to overwrite the instance (Y/N)? ").strip().lower() 

            if overwrite != 'y':
                print("Exiting without creating a new instance.")
                sys.exit()
        else:
            os.makedirs(instance_folder, exist_ok=True)

if is_baseline:
    output_csv_file = baseline_csv_file
    output_parquet_file = baseline_parquet_file
else:
    output_csv_file = instance_csv_file
    output_parquet_file = instance_parquet_file

all_dfs = []

# Loop through all JSON files in the specified folder
for json_path in glob.glob(os.path.join(pbi_pa_folder, r"*.json")):
    with open(json_path, "r", encoding="utf-8-sig") as f:
        data = json.load(f)
    events = data.get("events", [])
    id_map = {event['id']: event for event in events if 'id' in event}
    flat_events = [flatten_event(e) for e in events]
    df = pd.DataFrame(flat_events)
    if not df.empty:
        df['visualId'] = df['id'].apply(lambda x: find_visual_id(id_map, x))
        all_dfs.append(df)

# Combine all DataFrames into one
if all_dfs:
    final_df = pd.concat(all_dfs, ignore_index=True)
else:
    final_df = pd.DataFrame()

pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', 20)    

filtered_df = final_df[final_df['name'] == 'Execute DAX Query'].copy()
filtered_df.drop_duplicates(inplace=True)
filtered_df['result_sets'] = 0
filtered_df['combined_query_hash'] = ""

# Loop through the DataFrame and execute each DAX query
with Pyadomd(connection_string) as conn:
    for idx, row in filtered_df.iterrows():
        query_id = row['id']
        query_text = row['QueryText']

        try:
            with conn.cursor().execute(query_text) as cur:
                result_set_index = 0
                resultset_hashes = []  # List to collect all query hashes
                has_next = True
                while has_next:
                    result_set_index += 1
                    columns = [col[0] for col in cur.description]
                    result_rows = cur.fetchall()
                    # print(f"QueryID: {query_id} Result set {result_set_index} has {len(result_rows)} rows")

                    if result_rows:
                        # Create a DataFrame for this query's results
                        result_df = pd.DataFrame(result_rows, columns=columns)

                        # Add a hash column for each row
                        if not result_df.empty:
                            result_df['row_hash'] = result_df.apply(row_hash, axis=1)

                            # Sort the DataFrame by row_hash to ensure deterministic ordering
                            result_df = result_df.sort_values('row_hash').reset_index(drop=True)
                            
                            # Generate a combined hash for the result set
                            row_hashes = '|'.join(result_df['row_hash'].tolist())
                            combined_row_hash = hashlib.sha256(row_hashes.encode('utf-8')).hexdigest()
                            resultset_hashes.append(combined_row_hash)
                        else:
                            print(f"query: {query_id} result set {result_set_index} is empty")

                    # If there are more result sets for the same query, continue fetching
                    has_next = cur.nextresult()

                filtered_df.loc[idx, 'result_sets'] = result_set_index

                if resultset_hashes:
                    resultset_hashes.sort()
                    combined = '|'.join(resultset_hashes)
                    combined_query_hash = hashlib.sha256(combined.encode('utf-8')).hexdigest()
                    filtered_df.loc[idx, 'combined_query_hash'] = combined_query_hash
                else:
                    filtered_df.loc[idx, 'combined_query_hash'] = None
                    print(f"No result sets for query {query_id}")

        except Exception as e:
            print(f"Error executing query {query_id}: {e}\n")

# Drop NaN values and convert to string (in case)
all_hashes = filtered_df['combined_query_hash'].dropna().astype(str).tolist()
combined = '|'.join(all_hashes)
final_overall_hash = hashlib.sha256(combined.encode('utf-8')).hexdigest()
filtered_df['final_overall_hash'] = final_overall_hash

print(f"Final overall hash for all combined_query_hash values: {final_overall_hash}")

filtered_df = filtered_df[['id', 'parentId', 'visualId', 'QueryText', 'RowCount', 'result_sets', 'combined_query_hash', 'final_overall_hash']]

filtered_df.to_csv(output_csv_file, index=False)
filtered_df.to_parquet(output_parquet_file, index=False)

#If this run is not a baseline, then compare to the baseline
if baseline_exists and instance_name != baseline_folder_name:
    if not os.path.isfile(baseline_parquet_file):
        print("Baseline file does not exist. Please create a baseline first.")
        sys.exit(1)

    baseline_df = pd.read_parquet(baseline_parquet_file)

    # Compare the current filtered_df with the baseline_df
    comparison_df = filtered_df.merge(baseline_df, on='id', suffixes=('', '_baseline'), how='outer', indicator=True)

    # Identify value differences in columns for rows present in both
    cols_to_compare = ['combined_query_hash']
    diff_mask = (comparison_df['_merge'] == 'both') & (
        comparison_df[[col for col in cols_to_compare]].ne(
            comparison_df[[f"{col}_baseline" for col in cols_to_compare]].values
        ).any(axis=1)
    )

    value_diffs = comparison_df[diff_mask]
    # if not value_diffs.empty:
    #     print("Rows with differing values between current and baseline:")
    #     print(value_diffs[['id'] + [col for col in cols_to_compare] + [f"{col}_baseline" for col in cols_to_compare]])
    # else:
    #     print("No value differences in shared rows.")
    
    value_diffs = value_diffs[['id', 'parentId', 'visualId', 'QueryText', 'RowCount', 'result_sets', 'combined_query_hash', 'combined_query_hash_baseline', 'final_overall_hash', 'final_overall_hash_baseline']]

    add_page_names_to_df(value_diffs, pbi_report_folder)

    display(value_diffs)


Final overall hash for all combined_query_hash values: e739361488db2ed056cada7fd758541551f755925fd5a42420510c36fd143212


,id,parentId,visualId,QueryText,RowCount,result_sets,combined_query_hash,combined_query_hash_baseline,final_overall_hash,final_overall_hash_baseline,pageName
6,206771da-ca77-40ad-a75b-280d0d7a62eb,ba8a86ad-88b7-4d0e-a6cc-e76c8ae7479f,fd225c7a3a3a0ea6006c,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tTRE...,1.0,1,c0920e0606b87cfbe1feeccad15fd2a7bbbf93018bac52...,1c08ffdf06410e3a608a3c0ebb2d4a2c1862ff83a5f676...,e739361488db2ed056cada7fd758541551f755925fd5a4...,b21af9321776ba8db715cb1ae33094c428ce635e51dba8...,Net Sales
9,29e00e61-0e58-4c7d-b5a5-3d1fbd53060b,537dfec7-a312-41f3-a000-dcaba20fa0cb,bdbdc676ded10ad92c90,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tTRE...,1.0,1,6ace68543c2149a619567fd5634d946d0e03e8def3cc67...,4228adbc40ca81ef2a943ad490c22a4cd0098f0c45e4bc...,e739361488db2ed056cada7fd758541551f755925fd5a4...,b21af9321776ba8db715cb1ae33094c428ce635e51dba8...,Net Sales
12,3116d681-710e-4c4a-b34b-aa867a01d347,5466294c-f42e-4373-9c22-0dc27418be93,b33397810d555ca70a8c,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,16.0,2,f3a7f136a841baffbe6577eaf2758d7653b5786b6ba09c...,113805e845ab8d302bf3a430f549c710267e1bf81c3578...,e739361488db2ed056cada7fd758541551f755925fd5a4...,b21af9321776ba8db715cb1ae33094c428ce635e51dba8...,Return Rate
24,80d1289d-9fde-4a2a-83f7-7a1e6abfd856,69f38537-2166-447c-b254-f865c96bc6ff,805719ca6000cb000be2,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,27.0,1,774b1332783bc40bfe610839f567ef9a734b18fe294480...,9d249633ca25d0a143eedc3e988b0242502ca1d6e1b021...,e739361488db2ed056cada7fd758541551f755925fd5a4...,b21af9321776ba8db715cb1ae33094c428ce635e51dba8...,Return Rate
39,c8f382ac-e545-4879-bd35-67f20096b8d7,8e712962-ac31-485e-8c04-c0d576955b18,948988db07a4db09d58d,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,26.0,1,9bcaf8e5411523b21dd4e096d41eee979d58c5d171c28b...,0fea39528cc2f3bfc39d83d0a11d892a376f194ade4cf4...,e739361488db2ed056cada7fd758541551f755925fd5a4...,b21af9321776ba8db715cb1ae33094c428ce635e51dba8...,Return Rate
46,e033aaf7-a2aa-413f-8a2f-969a4deb4615,d96a4290-a4a6-4b4e-9ba8-7da7a725cb34,e5b7b6b90231b41809a1,DEFINE\r\n\tVAR __DS0FilterTable = \r\n\t\tFIL...,1.0,1,4eaca420673e3994a1748aa6bae951e616ad9d4a91861e...,a3e30f2f2b65e66d1016efa882c59db7b1b8cf50503758...,e739361488db2ed056cada7fd758541551f755925fd5a4...,b21af9321776ba8db715cb1ae33094c428ce635e51dba8...,Return Rate
